In [1]:
import pandas as pd
import pyarrow
import os
import geopandas as gpd
import folium
import requests
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import snowflake.connector
import mlflow
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
import lightgbm as lgb
import xgboost as xgb
import optuna
from optuna.integration.mlflow import MLflowCallback

os.environ["GIT_PYTHON_REFRESH"] = "quiet"

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Connect to Snowflake

In [2]:
load_dotenv()
conn = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    warehouse="compute_wh",
    database="nyc_taxi",
    schema="surge_predictor"
)

In [3]:
cs = conn.cursor()

Sanity Check

In [4]:
cs.execute('DESCRIBE TABLE "fct_zone_features";')
for row in cs.fetchall():
    print(row[0])  # column name is first field

ZONE_ID
PICKUP_HOUR_TS
PICKUP_COUNT
PICKUP_LAG_1H
PICKUP_LAG_2H
PICKUP_LAG_3H
ROLLING_AVG_3H
ROLLING_AVG_24H
DAY_OF_WEEK
IS_WEEKEND
HOUR_OF_DAY
IS_HOLIDAY


In [5]:
cs.execute('SELECT * FROM "fct_zone_features" LIMIT 5;')
print([col[0] for col in cs.description])

['ZONE_ID', 'PICKUP_HOUR_TS', 'PICKUP_COUNT', 'PICKUP_LAG_1H', 'PICKUP_LAG_2H', 'PICKUP_LAG_3H', 'ROLLING_AVG_3H', 'ROLLING_AVG_24H', 'DAY_OF_WEEK', 'IS_WEEKEND', 'HOUR_OF_DAY', 'IS_HOLIDAY']


In [6]:
def run_query(sql):
    cs.execute(sql)
    cols = [c[0] for c in cs.description]
    return pd.DataFrame(cs.fetchall(), columns=cols)

In [7]:
# Zero-fill check, 1 for EWR Newark Airport
cs.execute("""
SELECT ZONE_ID, PICKUP_HOUR_TS, PICKUP_COUNT
FROM "fct_zone_features"
WHERE ZONE_ID = 1
ORDER BY PICKUP_HOUR_TS
LIMIT 48;
""")
cols = [c[0] for c in cs.description]
df = pd.DataFrame(cs.fetchall(), columns=cols)
df

,ZONE_ID,PICKUP_HOUR_TS,PICKUP_COUNT
0,1,2026-01-01 00:00:00,0
1,1,2026-01-01 01:00:00,0
2,1,2026-01-01 02:00:00,0
3,1,2026-01-01 03:00:00,0
4,1,2026-01-01 04:00:00,0
5,1,2026-01-01 05:00:00,0
6,1,2026-01-01 06:00:00,0
7,1,2026-01-01 07:00:00,0
8,1,2026-01-01 08:00:00,0
9,1,2026-01-01 09:00:00,0


In [8]:
# Lag alignment, 161 for Manhattan
sql="""SELECT ZONE_ID, PICKUP_HOUR_TS, PICKUP_COUNT,
       PICKUP_LAG_1H, PICKUP_LAG_2H, PICKUP_LAG_3H
FROM "fct_zone_features"
WHERE ZONE_ID = 161 
ORDER BY PICKUP_HOUR_TS
LIMIT 30;"""

run_query(sql)


,ZONE_ID,PICKUP_HOUR_TS,PICKUP_COUNT,PICKUP_LAG_1H,PICKUP_LAG_2H,PICKUP_LAG_3H
0,161,2026-01-01 00:00:00,376,NaN,NaN,NaN
1,161,2026-01-01 01:00:00,343,376.0,NaN,NaN
2,161,2026-01-01 02:00:00,275,343.0,376.0,NaN
3,161,2026-01-01 03:00:00,208,275.0,343.0,376.0
4,161,2026-01-01 04:00:00,103,208.0,275.0,343.0
5,161,2026-01-01 05:00:00,61,103.0,208.0,275.0
6,161,2026-01-01 06:00:00,50,61.0,103.0,208.0
7,161,2026-01-01 07:00:00,46,50.0,61.0,103.0
8,161,2026-01-01 08:00:00,61,46.0,50.0,61.0
9,161,2026-01-01 09:00:00,86,61.0,46.0,50.0


In [9]:
# Rolling average correctness
sql="""
SELECT ZONE_ID, PICKUP_HOUR_TS, PICKUP_COUNT,
       ROLLING_AVG_3H, ROLLING_AVG_24H
FROM "fct_zone_features"
WHERE ZONE_ID = 161
ORDER BY PICKUP_HOUR_TS
LIMIT 30;
"""
run_query(sql)

,ZONE_ID,PICKUP_HOUR_TS,PICKUP_COUNT,ROLLING_AVG_3H,ROLLING_AVG_24H
0,161,2026-01-01 00:00:00,376,None,None
1,161,2026-01-01 01:00:00,343,376.000,376.000
2,161,2026-01-01 02:00:00,275,359.500,359.500
3,161,2026-01-01 03:00:00,208,331.333,331.333
4,161,2026-01-01 04:00:00,103,275.333,300.500
5,161,2026-01-01 05:00:00,61,195.333,261.000
6,161,2026-01-01 06:00:00,50,124.000,227.666
7,161,2026-01-01 07:00:00,46,71.333,202.285
8,161,2026-01-01 08:00:00,61,52.333,182.750
9,161,2026-01-01 09:00:00,86,52.333,169.222


In [10]:
# Holiday flag check
sql="""
SELECT PICKUP_HOUR_TS, IS_HOLIDAY
FROM "fct_zone_features"
WHERE PICKUP_HOUR_TS::date IN ('2026-01-01', '2026-01-19')
LIMIT 10;
"""
run_query(sql)

,PICKUP_HOUR_TS,IS_HOLIDAY
0,2026-01-01 00:00:00,1
1,2026-01-01 01:00:00,1
2,2026-01-01 02:00:00,1
3,2026-01-01 03:00:00,1
4,2026-01-01 04:00:00,1
5,2026-01-01 05:00:00,1
6,2026-01-01 06:00:00,1
7,2026-01-01 07:00:00,1
8,2026-01-01 08:00:00,1
9,2026-01-01 09:00:00,1


In [11]:
# Early-hours edge case
sql="""
SELECT ZONE_ID, PICKUP_COUNT,PICKUP_HOUR_TS, PICKUP_LAG_1H, PICKUP_LAG_2H, PICKUP_LAG_3H, ROLLING_AVG_24H
FROM "fct_zone_features"
WHERE ZONE_ID = 161
ORDER BY PICKUP_HOUR_TS
LIMIT 5;
"""
run_query(sql)

,ZONE_ID,PICKUP_COUNT,PICKUP_HOUR_TS,PICKUP_LAG_1H,PICKUP_LAG_2H,PICKUP_LAG_3H,ROLLING_AVG_24H
0,161,376,2026-01-01 00:00:00,NaN,NaN,NaN,None
1,161,343,2026-01-01 01:00:00,376.0,NaN,NaN,376.000
2,161,275,2026-01-01 02:00:00,343.0,376.0,NaN,359.500
3,161,208,2026-01-01 03:00:00,275.0,343.0,376.0,331.333
4,161,103,2026-01-01 04:00:00,208.0,275.0,343.0,300.500


### Model training and MLflow tracking

In [12]:
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("nyc_surge_gbdt")

<Experiment: artifact_location='/mlruns/1', creation_time=1786249107704, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786249107704, lifecycle_stage='active', name='nyc_surge_gbdt', tags={}, trace_location=None, workspace='default'>

In [13]:
cs.execute('SELECT * FROM "fct_zone_features" ORDER BY ZONE_ID, PICKUP_HOUR_TS;')
cols = [c[0] for c in cs.description]
df = pd.DataFrame(cs.fetchall(), columns=cols)

In [14]:
# Restrict to January only for train/val — Feb is held out entirely for final evaluation
df_jan = df[df["PICKUP_HOUR_TS"] < "2026-02-01"].copy()
print(df_jan.shape)  # expect (265*744, 12) = (197160, 12)

(197160, 12)


In [15]:
for col in ["ROLLING_AVG_3H", "ROLLING_AVG_24H"]:
    df_jan[col] = df_jan[col].astype(float)

In [16]:
split_time = df_jan["PICKUP_HOUR_TS"].quantile(0.8)
train = df_jan[df_jan["PICKUP_HOUR_TS"] <= split_time]
val = df_jan[df_jan["PICKUP_HOUR_TS"] > split_time]

Using GBDT (Gradient Boosted Decision Tree) Model

In [17]:
features = ["PICKUP_LAG_1H","PICKUP_LAG_2H","PICKUP_LAG_3H",
            "ROLLING_AVG_3H","ROLLING_AVG_24H",
            "DAY_OF_WEEK","IS_WEEKEND","HOUR_OF_DAY","IS_HOLIDAY"]
target = "PICKUP_COUNT"

X_train, y_train = train[features], train[target]
X_val, y_val = val[features], val[target]

with mlflow.start_run(run_name="gbdt_baseline"):
    model = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    mlflow.log_params({"model": "hist_gbdt", "max_iter": 200, "learning_rate": 0.05, "max_depth": 5})
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(model, "model")

2026/08/29 20:41:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run gbdt_baseline at: http://mlflow:5000/#/experiments/1/runs/37f4233d62f144ef835dd817664e90db
🧪 View experiment at: http://mlflow:5000/#/experiments/1


Training LightGBM and XGBoost

In [18]:
with mlflow.start_run(run_name="lightgbm_baseline"):
    model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42, verbose=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    mlflow.log_params({"model": "lightgbm", "n_estimators": 200, "learning_rate": 0.05, "max_depth": 5})
    mlflow.log_metric("rmse", rmse)
    mlflow.lightgbm.log_model(model, "model")

with mlflow.start_run(run_name="xgboost_baseline"):
    model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    mlflow.log_params({"model": "xgboost", "n_estimators": 200, "learning_rate": 0.05, "max_depth": 5})
    mlflow.log_metric("rmse", rmse)
    mlflow.xgboost.log_model(model, "model")

2026/08/29 20:42:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run lightgbm_baseline at: http://mlflow:5000/#/experiments/1/runs/c4690139759640e79e4bf9780462c57b
🧪 View experiment at: http://mlflow:5000/#/experiments/1


2026/08/29 20:42:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run xgboost_baseline at: http://mlflow:5000/#/experiments/1/runs/1cd580609fe54f259a5450423054b9a8
🧪 View experiment at: http://mlflow:5000/#/experiments/1


In [19]:
df["PICKUP_COUNT"].describe()

count    375240.000000
mean        108.471631
std         120.975951
min           0.000000
25%          24.000000
50%          73.000000
75%         153.000000
max        1846.000000
Name: PICKUP_COUNT, dtype: float64

Naive Model Baseline (predicting pickup count based on 1H lag)

In [20]:
naive_preds = val["PICKUP_LAG_1H"]
naive_rmse = np.sqrt(mean_squared_error(val["PICKUP_COUNT"], naive_preds))
print("Naive (lag-1h) RMSE:", naive_rmse)

with mlflow.start_run(run_name="naive_baseline"):
    mlflow.log_param("model", "naive_lag1h")
    mlflow.log_metric("rmse", naive_rmse)

Naive (lag-1h) RMSE: 35.64176039620861
🏃 View run naive_baseline at: http://mlflow:5000/#/experiments/1/runs/62b137333d664eacbdf996d447d7b43b
🧪 View experiment at: http://mlflow:5000/#/experiments/1


Hyper-parameter Tuning using Optuna 

In [21]:
mlflc = MLflowCallback(
    tracking_uri="http://mlflow:5000",
    metric_name="rmse"
)


@mlflc.track_in_mlflow()
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "random_state": 42,
        "verbose": -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    return rmse

study = optuna.create_study(direction="minimize",study_name="lightgbm_tuning")
study.optimize(objective, n_trials=30, callbacks=[mlflc])

print("Best RMSE:", study.best_value)
print("Best params:", study.best_params)

/tmp/ipykernel_2372/279063426.py:1: FutureWarning: MLflowCallback has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0.
  mlflc = MLflowCallback(
/tmp/ipykernel_2372/279063426.py:7: ExperimentalWarning: optuna_integration.mlflow.mlflow.MLflowCallback.track_in_mlflow is experimental (supported from v2.9.0). The interface can change in the future.
  @mlflc.track_in_mlflow()
[I 2026-08-29 20:42:08,409] A new study created in memory with name: lightgbm_tuning
[I 2026-08-29 20:42:08,924] Trial 0 finished with value: 26.34648736590125 and parameters: {'n_estimators': 300, 'learning_rate': 0.18770643916688676, 'max_depth': 3, 'num_leaves': 76}. Best is trial 0 with value: 26.34648736590125.


🏃 View run 0 at: http://mlflow:5000/#/experiments/2/runs/7e7f0aae7e5f46f3b3b0292aa2b196c4
🧪 View experiment at: http://mlflow:5000/#/experiments/2
🏃 View run 0 at: http://mlflow:5000/#/experiments/2/runs/7e7f0aae7e5f46f3b3b0292aa2b196c4
🧪 View experiment at: http://mlflow:5000/#/experiments/2


[I 2026-08-29 20:42:10,340] Trial 1 finished with value: 26.927586993921018 and parameters: {'n_estimators': 500, 'learning_rate': 0.013434570066417803, 'max_depth': 4, 'num_leaves': 35}. Best is trial 0 with value: 26.34648736590125.


🏃 View run 1 at: http://mlflow:5000/#/experiments/2/runs/4989a6dae3c443e8b05ff3ee13ec4107
🧪 View experiment at: http://mlflow:5000/#/experiments/2
🏃 View run 1 at: http://mlflow:5000/#/experiments/2/runs/4989a6dae3c443e8b05ff3ee13ec4107
🧪 View experiment at: http://mlflow:5000/#/experiments/2


[I 2026-08-29 20:42:11,409] Trial 2 finished with value: 27.283660709740538 and parameters: {'n_estimators': 100, 'learning_rate': 0.03143152110753428, 'max_depth': 8, 'num_leaves': 61}. Best is trial 0 with value: 26.34648736590125.


🏃 View run 2 at: http://mlflow:5000/#/experiments/2/runs/deac54412b8746508b72620db7cb191f
🧪 View experiment at: http://mlflow:5000/#/experiments/2
🏃 View run 2 at: http://mlflow:5000/#/experiments/2/runs/deac54412b8746508b72620db7cb191f
🧪 View experiment at: http://mlflow:5000/#/experiments/2


... (50 outputs left out)

[I 2026-08-29 20:42:51,180] Trial 28 finished with value: 25.886029489804358 and parameters: {'n_estimators': 350, 'learning_rate': 0.050223929596448225, 'max_depth': 7, 'num_leaves': 57}. Best is trial 3 with value: 25.79281275179558.


🏃 View run 28 at: http://mlflow:5000/#/experiments/2/runs/0cd04d9604cc4c359adf4eaabddddfa5
🧪 View experiment at: http://mlflow:5000/#/experiments/2
🏃 View run 28 at: http://mlflow:5000/#/experiments/2/runs/0cd04d9604cc4c359adf4eaabddddfa5
🧪 View experiment at: http://mlflow:5000/#/experiments/2


[I 2026-08-29 20:42:52,564] Trial 29 finished with value: 25.93157087137827 and parameters: {'n_estimators': 250, 'learning_rate': 0.06622726566542454, 'max_depth': 10, 'num_leaves': 78}. Best is trial 3 with value: 25.79281275179558.


🏃 View run 29 at: http://mlflow:5000/#/experiments/2/runs/fb97119e3aea4990a6f357f196bf999a
🧪 View experiment at: http://mlflow:5000/#/experiments/2
🏃 View run 29 at: http://mlflow:5000/#/experiments/2/runs/fb97119e3aea4990a6f357f196bf999a
🧪 View experiment at: http://mlflow:5000/#/experiments/2
Best RMSE: 25.79281275179558
Best params: {'n_estimators': 400, 'learning_rate': 0.035853669553879086, 'max_depth': 9, 'num_leaves': 41}


Final Model

In [22]:
with mlflow.start_run(run_name="lightgbm_tuned_final"):
    final_params = {
        "n_estimators": 400,
        "learning_rate": 0.04812572188914544,
        "max_depth": 8,
        "num_leaves": 98,
        "random_state": 42,
        "verbose": -1,
    }
    model = lgb.LGBMRegressor(**final_params)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))

    mlflow.log_params({"model": "lightgbm_tuned", **final_params})
    mlflow.log_metric("rmse", rmse)
    mlflow.lightgbm.log_model(model, "model")

    print("Final tuned RMSE:", rmse)

2026/08/29 20:42:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Final tuned RMSE: 25.816715391551625
🏃 View run lightgbm_tuned_final at: http://mlflow:5000/#/experiments/2/runs/f6a3692db0a9431e873d19ab73d8b06f
🧪 View experiment at: http://mlflow:5000/#/experiments/2


In [23]:
cs.execute('SELECT COUNT(*) FROM "fct_zone_features";')
print(cs.fetchone())

(375240,)


## Model evaluation on unseen (February) Data

In [24]:
cs.execute("""
SELECT *
FROM "fct_zone_features"
WHERE PICKUP_HOUR_TS >= '2026-02-01'
ORDER BY ZONE_ID, PICKUP_HOUR_TS;
""")
cols = [c[0] for c in cs.description]
df_feb_eval = pd.DataFrame(cs.fetchall(), columns=cols)
print(df_feb_eval.shape)  # expect (178080, 12)

(178080, 12)


In [25]:
for col in ["ROLLING_AVG_3H", "ROLLING_AVG_24H"]:
    df_feb_eval[col] = df_feb_eval[col].astype(float)

In [26]:
features = ["PICKUP_LAG_1H","PICKUP_LAG_2H","PICKUP_LAG_3H",
            "ROLLING_AVG_3H","ROLLING_AVG_24H",
            "DAY_OF_WEEK","IS_WEEKEND","HOUR_OF_DAY","IS_HOLIDAY"]
target = "PICKUP_COUNT"

X_feb = df_feb_eval[features]
y_feb = df_feb_eval[target]

feb_preds = model.predict(X_feb)  # reuse the `model` object from lightgbm_tuned_final
feb_rmse = np.sqrt(mean_squared_error(y_feb, feb_preds))
print("Feb (out-of-sample) RMSE:", feb_rmse)

Feb (out-of-sample) RMSE: 26.176480315853706


In [28]:
with mlflow.start_run(run_name="lightgbm_tuned_feb_eval"):
    mlflow.log_param("eval_set", "february_holdout")
    mlflow.log_metric("rmse", feb_rmse)

🏃 View run lightgbm_tuned_feb_eval at: http://mlflow:5000/#/experiments/2/runs/8e388858495e4ded89e2d9d95991e340
🧪 View experiment at: http://mlflow:5000/#/experiments/2


In [29]:
# Confirm train is Jan-only
print(train["PICKUP_HOUR_TS"].min(), train["PICKUP_HOUR_TS"].max())
print(val["PICKUP_HOUR_TS"].min(), val["PICKUP_HOUR_TS"].max())
# Both should fall entirely within January

2026-01-01 00:00:00 2026-01-25 19:00:00
2026-01-25 20:00:00 2026-01-31 23:00:00
